# Maracatu-20M: Brazilian Portuguese Language Model 🥁

**Maracatu-20M** is a 17M-parameter causal language model trained from scratch on Brazilian Portuguese Wikipedia.
It is the first public checkpoint of the [Maracatu AI](https://github.com/maracatu-ai/maracatu) project, an open effort to build Portuguese-language LLMs with full transparency over architecture, data, and training.

Architecture: Llama-style decoder-only transformer (RMSNorm, RoPE, SwiGLU), compatible with `AutoModelForCausalLM` from HuggingFace Transformers.
Tokenizer: SentencePiece BPE, 16k vocabulary, trained on Brazilian Portuguese text.
Context window: 512 tokens.

This is a **base model** (text completion only). It is not an instruction-following or chat assistant.

Full model card: [huggingface.co/maracatu-ai/maracatu-20m](https://huggingface.co/maracatu-ai/maracatu-20m)
Source code and training details: [github.com/maracatu-ai/maracatu](https://github.com/maracatu-ai/maracatu)
License: Apache 2.0


In [ ]:
import kagglehub
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Download model weights and tokenizer files
model_path = kagglehub.model_download("whereisanzi/maracatu-20m/transformers/default")
print("Model downloaded to:", model_path)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# use_fast=False is required because this model ships with a SentencePiece
# tokenizer.model (no tokenizer.json). LlamaTokenizerFast would misload it.
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float32)
model = model.to(device)
model.eval()

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
def generate(prompt, max_new_tokens=80, temperature=0.8, top_k=50, seed=42):
    """Generate text continuation from a prompt."""
    torch.manual_seed(seed)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature,
            top_k=top_k,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


## Example 1: simple completion

The model continues a short seed phrase. Output is lowercase because the tokenizer normalizes all input to lowercase (`nmt_nfkc_cf` normalization).


In [ ]:
prompt_1 = "O Brasil e"
output_1 = generate(prompt_1, max_new_tokens=80, temperature=0.8, top_k=50, seed=42)
print("Prompt :", prompt_1)
print("Output :", output_1)


## Example 2: literary prompt

A prompt grounded in Brazilian literary history. At 17M parameters, the model has limited factual reliability, but syntactic structure and Portuguese fluency are consistent.


In [ ]:
prompt_2 = "Machado de Assis escreveu"
output_2 = generate(prompt_2, max_new_tokens=80, temperature=0.8, top_k=50, seed=42)
print("Prompt :", prompt_2)
print("Output :", output_2)


## Example 3: comparing sampling temperatures

The `temperature` parameter controls the sharpness of the probability distribution over the vocabulary at each step.

- Low temperature (0.3): more deterministic, repetitive but coherent.
- Mid temperature (0.8): balanced diversity and fluency.
- High temperature (1.2): more creative, higher risk of incoherence.


In [ ]:
prompt_3 = "A cidade de Sao Paulo"
temperatures = [0.3, 0.8, 1.2]

for temp in temperatures:
    output = generate(prompt_3, max_new_tokens=60, temperature=temp, top_k=50, seed=42)
    print(f"--- temperature={temp} ---")
    print(output)
    print()


## Limitations

**Scale.** At 17M parameters, Maracatu-20M is a research checkpoint, not a production model. Larger checkpoints (500M, 1B, 7B) are planned as the project matures.

**Factual hallucination.** The model frequently generates plausible-sounding but incorrect facts. Do not use outputs as factual references.

**Lowercase output.** The tokenizer applies `nmt_nfkc_cf` normalization, which folds everything to lowercase. Proper nouns and sentence-initial capitalization are lost.

**Base model only.** This checkpoint does not follow instructions or answer questions. It continues text. Feeding it questions will produce continuations, not answers.

**Training data.** Corpus v1 is Brazilian Portuguese Wikipedia only (approximately 550M tokens). Coverage of specialized domains (law, medicine, code) is limited.

**Context window.** Maximum context is 512 tokens. Inputs longer than that will be truncated.


## Next steps

Maracatu is an ongoing project. Upcoming checkpoints will cover larger parameter counts, expanded corpora, and benchmark evaluations on Brazilian Portuguese tasks (ENEM, OAB, BLUEX, POSCOMP).

Resources:

- HuggingFace Hub (weights, model card, full config): [huggingface.co/maracatu-ai/maracatu-20m](https://huggingface.co/maracatu-ai/maracatu-20m)
- Kaggle Model page: [kaggle.com/models/whereisanzi/maracatu-20m](https://www.kaggle.com/models/whereisanzi/maracatu-20m)
- Source code, training scripts, architecture docs: [github.com/maracatu-ai/maracatu](https://github.com/maracatu-ai/maracatu)
- License: Apache 2.0
